# Cetacea Classifier — Master Notebook

**Authors:** Rafael G. de Menezes & Alexandre Paro  
**Reference:** Paro et al., 2025 (in preparation)

---

### 🎯 Objective
Development of a **meta-classifier for cetacean species** using PAMGuard ROCCA master sheets.  
This notebook integrates multiple training and evaluation strategies:

- **K-FOLD**: cross-validation across samples  
- **LOGO-EVENT**: Leave-One-Group-Out by event  
- **LOGO-INDIVIDUAL**: Leave-One-Group-Out by individual  
- **META**: ensemble/meta-models  
- **New Data Prediction**: applying trained models to unseen datasets  

---

### 📦 Outputs
The workflow generates reproducible and shareable artifacts:

- **Feature tables** (processed and standardized)  
- **Trained model(s)** ready for reuse  
- **Evaluation materials**: plots, metrics, and summary reports  


## Importing libraries and setting variables

In [ ]:
sep = '=' * 50
print(f"{sep}\nCetacea Whistle Classifier — Master Notebook\nStarting code, imports, and global variables")
import os, platform, warnings
from typing import Iterable, Tuple, List, Dict, Any, Optional
from pathlib import Path
# from itertools import chain
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, accuracy_score
# from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as W
from IPython.display import display
print("\tPython:", platform.python_version())
print("\tNumPy:", np.__version__)
print("\tPandas:", pd.__version__)
print("\tScikit-learn:", sklearn.__version__)
pd.set_option('display.max_rows', None )
pd.set_option('display.max_columns', None )
pd.set_option('display.width', 100)
print(sep)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
print_fold_stats = True # change to False to ommit each fold individual statistics

BASE_DIR = Path.cwd()
DATA_DIR = os.path.join(BASE_DIR, "inputs")
OUT_DIR  =  os.path.join(BASE_DIR, "outputs")
inputs_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
name_box = W.Text(value="baseline", description="Run name:")
file_dropdown = W.Dropdown(options=inputs_files, description="Training input file:")
file_dropdown = W.Dropdown(options=inputs_files, description="Training input file:")
display(W.VBox([name_box, file_dropdown]))

# --- GENERAL FUNCTIONS ---
def read_ROCCA_sheet(filepath, type = 'original'):
    df = pd.read_csv(filepath)
    if type != 'original':
        df = df.drop(['Source', 'EncounterID', 'CruiseID'], axis=1)
    else:
        col_exclude = (list(df.loc[:, 'Source':'EncounterCount'].columns) +
                        list(df.loc[:, 'SamplingRate':'DataProvider'].columns) + 
                        ['GeographicLocation','ClassifiedSpecies', 'DCMEAN', 'DCSTDDEV'] +
                        list(df.loc[:, 'DCQUARTER1MEAN':'DCQUARTER4MEAN'].columns) +
                        list(df.loc[:, 'FREQBEGSWEEP':'FREQENDDWN'].columns) +
                        list(df.loc[:, 'FREQPEAK':'Species_List'].columns))
        df = df.drop(columns = col_exclude)
    return df

def get_preset_configs (PRESETS, PATH, preset_label, grouping_type='LOGO'):
    if preset_label not in PRESETS["preset_labels"]:
        raise ValueError(f"Preset label {preset_label!r} not found in PRESETS['preset_labels'].")
    i = PRESETS['preset_labels'].index(preset_label)
    configs = {k: v[i] for k, v in PRESETS.items() if k != "preset_labels"} # Build a dict with the i-th value for each key

    configs['oob_score'] = True
    if  grouping_type.upper() == 'LOGO':
        configs['n_jobs'] = -1
    elif  grouping_type.upper() in ['KFOLD','FOLD']:
        configs['min_samples_leaf'] = 1
        configs['min_samples_split'] = 2

    cfgfile_path = os.path.join(PATH, f'{preset_label}_{grouping_type}_configs.txt')
    with open(cfgfile_path, 'w') as f:
        print("Configuration:")
        for key, value in configs.items():
            f.write(f"{key}: {value}\n")
            print(f"\t{key:>12}: {value}")

    rfc_params = {k: v for k, v in configs.items() if not k.startswith("rs_")}
    return configs, rfc_params
    
def adjust_confusion_matrix(matrix): 
    # Function to round and adjust the confusion matrix percentages that sum up to 100% in each row
    matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis] # Compute the confusion matrix percentage
    rounded_matrix = np.floor(matrix * 100).astype(int)  # Round down to ensure totals <= 100
    row_sums = rounded_matrix.sum(axis=1)
    for i in range(len(row_sums)):
        errors = 100 - row_sums[i]  # Calculate how much we need to adjust
        if errors > 0:
            # Increment the highest decimal part entries
            decimal_part = (matrix[i] * 100) - np.floor(matrix[i] * 100)
            adjust_indices = np.argsort(decimal_part)[-errors:]  # Get indices with the highest decimals
            rounded_matrix[i, adjust_indices] += 1
    return rounded_matrix

def plot_confusion_matrix_heatmap(matrix, class_labels: List[str], preset: str, 
                            save_dir: str, sorted: bool=True, grouping: str = "LOGO", show: bool = False, dpi: int = 300):
    """ Plot and save a styled confusion matrix heatmap.
    Parameters
        matrix : np.ndarray ->             Confusion matrix (integer percentages, e.g. from adjust_confusion_matrix).
        class_labels : list of str ->       Labels for rows/columns.
        preset : str ->                      Preset name (used in title and filename).
        save_dir : str ->                    Directory to save the figure.
        sorted : bool, default=True ->       If True, sort classes by diagonal values.
        grouping : str, default="KFOLD" ->   Name to include in the title/filename.
        show : bool, default=False ->        If True, display with plt.show(); otherwise close after saving.
        dpi : int, default=300 ->           Resolution of saved PNG.
    Returns
        conf_figpath : str ->               Path where the figure was saved."""
    
    if sorted:    
        diagonal_values = np.diag(matrix)                          # Calculate diagonal values for sorting
        sorted_indices  = np.argsort(diagonal_values)[::-1]        # Get sorted indices based on diagonal values (highest to lowest)
        matrix          = matrix[sorted_indices][:, sorted_indices] # Create a new confusion matrix that reflects the sorted labels while preserving original percentages
        class_labels    = [class_labels[i] for i in sorted_indices] # Reorder the class labels according to the sorted diagonal values
        
    plt.figure()
    plt.title(f"{grouping} Confusion Matrix - {preset}")
    heatmap = sns.heatmap(matrix / 100.0,  # normalize to [0,1] for colormap
                          annot=True,cmap="Greys", fmt="", xticklabels=class_labels, yticklabels=class_labels,
                          annot_kws={"size": 10},cbar_kws={"shrink": 0.8}, linewidths=0.0,  linecolor="white",)
    # Style axes
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.3)
    # Style colorbar
    cbar = heatmap.collections[0].colorbar
    cbar.outline.set_edgecolor("black")
    cbar.outline.set_linewidth(0.2)
    # Axis labels
    plt.xlabel("Predicted Species", fontsize=12, fontweight="bold", labelpad=10)
    plt.ylabel("True Species", fontsize=12, fontweight="bold", labelpad=10)
    plt.xticks(rotation=45)
    # Save
    os.makedirs(save_dir, exist_ok=True)
    conf_figpath = os.path.join(save_dir, f"{preset}_{grouping}_Confusion_Matrix.png")
    print(f"\n[saving] Confusion Matrix {grouping} at {conf_figpath}")
    plt.savefig(conf_figpath, format="png", bbox_inches="tight", dpi=dpi)
    if show: plt.show()
    else:    plt.close()
    return conf_figpath

def average_presets_probs(tables: List[pd.DataFrame],presets_labels: List[str],
                        out_dir: str, method_tag: str = "KFOLD",
                        meta_cols: Tuple[str, ...] = ("Species_Classification", "UID", "EncounterNumber", "KnownSpecies"),
                        save: bool = True,) -> pd.DataFrame:
    """Average species probability columns across a list of DataFrames (one per preset).
    - Detects class-probability columns from the FIRST table as all columns
      before the first meta column in `meta_cols` (or all non-meta columns if
      no meta column is present).
    - Preserves class column order from the FIRST table.
    - Averages per UID if present; otherwise averages by row index.
    - Reattaches EncounterNumber & KnownSpecies from the first occurrence per UID/index.
    Returns the averaged DataFrame and optionally saves it as:
      _<concatenated preset labels>_<method_tag>_Averaged_Probs.csv"""
    if not tables: raise ValueError("No tables provided.")

    print('\nAveraging probabilities...')
    base = tables[0].copy()
    # Figure out class-prob columns (prefer “before first meta col” rule)
    cols = list(base.columns)
    stop_idx_candidates = [cols.index(c) for c in meta_cols if c in cols]
    if stop_idx_candidates:
        stop_idx = min(stop_idx_candidates)
        class_cols = cols[:stop_idx]
        # other_cols = cols[stop_idx:]
    else:
        class_cols = [c for c in cols if c not in meta_cols]
        # other_cols = [c for c in cols if c in meta_cols]
    if not class_cols:
        raise ValueError("No class-probability columns detected; check table layout/meta_cols.")

    # Concatenate all presets
    all_df = pd.concat(tables, ignore_index=True)
    # Average numeric class columns
    avg_probs = (all_df.groupby("UID", as_index=False)[class_cols].mean(numeric_only=True))
    # Take first EncounterNumber / KnownSpecies per UID
    meta_first = (all_df.groupby("UID", as_index=False)[[c for c in meta_cols if c != "UID"]].first())
    avg_probs = avg_probs.merge(meta_first, on="UID", how="left")
    # Reorder columns: class probs (in base order) + meta (UID, EncounterNumber, KnownSpecies in that order)
    meta_in_out = [c for c in meta_cols if c in avg_probs.columns]
    avg_probs = avg_probs[class_cols + meta_in_out]
    if save: # Save
        os.makedirs(out_dir, exist_ok=True)
        presets_str = "".join(presets_labels)
        fname = f"_{presets_str}_{method_tag}_Averaged_Probs.csv"
        fpath = os.path.join(out_dir, fname)
        print(f"[saving] {fpath}")
        avg_probs.to_csv(fpath, index=False)
    return avg_probs

# --- KFOLD FUNCTIONS ---
def balance_train_data_kfold(X, weights, target_class_size, RS_bal_1=42, RS_bal_2=43, RS_bal_3=44):
    all_sampled_data = pd.DataFrame()
    # Iterate over each class and their corresponding group weights
    for class_label, group_weights in weights.items():
        class_data = X[X['KnownSpecies'] == class_label]
        total_samples = 0
        group_samples = {}
        # First pass: Sample from groups according to their weights
        for group, weight in group_weights.items():
            group_data = class_data[class_data['EncounterNumber'] == group]
            n_samples = int(weight * target_class_size)  # Allocate samples based on group weight
            # Adjust n_samples if it exceeds available samples in the group
            n_samples = min(n_samples, len(group_data))
            total_samples += n_samples
            if n_samples > 0:
                group_samples[group] = group_data.sample(n=n_samples, replace=False, random_state= RS_bal_1)

        # Combine sampled groups for this class
        sampled_data    = pd.concat(group_samples.values())
        sampled_indices = sampled_data.index.tolist()  # Track sampled indices
        # If we don't have enough samples for this class, sample more from the remaining class data
        if len(sampled_data) < target_class_size:
            remaining_data = class_data[~class_data.index.isin(sampled_indices)]
            if not remaining_data.empty:
                # Calculate the number of additional samples needed
                additional_samples = target_class_size - len(sampled_data)
                if additional_samples > 0:
                    # Sample remaining data to fill the gap, ensuring we don't oversample any group
                    remaining_sampled_data = remaining_data.sample(n=additional_samples, replace=False, random_state= RS_bal_2)
                    sampled_data = pd.concat([sampled_data, remaining_sampled_data])
        # Ensure the final sample size matches the target_class_size
        if len(sampled_data) > target_class_size:
            sampled_data = sampled_data.sample(n=target_class_size, replace=False, random_state= RS_bal_3)
        # Accumulate the sampled data across all classes
        all_sampled_data = pd.concat([all_sampled_data, sampled_data])

    # Separate features and target for the final balanced dataset
    all_sampled_data = all_sampled_data.reset_index(drop=True)
    X_balanced = all_sampled_data.drop(columns=['KnownSpecies'])
    y_balanced = all_sampled_data['KnownSpecies'].reset_index(drop=True)
    return X_balanced, y_balanced

def predict_kfold_probs(new_data_clean, all_models_kfold):
    all_avg_probs = []

    for model_label, models in all_models_kfold.items():
        fold_probs = []

        for rf_model in models:
            X_new_input = new_data_clean.drop(columns=['EncounterNumber', 'KnownSpecies'], errors='ignore')
            probs = rf_model.predict_proba(X_new_input)
            probs_df = pd.DataFrame(probs, columns=rf_model.classes_, index=new_data_clean.index)
            fold_probs.append(probs_df)

        model_avg_probs = sum(fold_probs) / len(fold_probs)
        all_avg_probs.append(model_avg_probs)

    final_avg_probs_df_kfold = sum(all_avg_probs) / len(all_avg_probs)
    final_avg_probs_df_kfold.index = new_data_clean.index
    final_avg_probs_df_kfold.columns = [f'{col}_KFOLD_5' for col in final_avg_probs_df_kfold.columns]
    return final_avg_probs_df_kfold

# --- LOGO FUNCTIONS ---
def find_minimum(df: pd.DataFrame,factors: Iterable[str], print_details: bool = False) -> Tuple[int, pd.Index]:
    """Returns:
        nbal: minimum count per factor1
        factor1_list: ordered unique labels of factor1 (Index)"""
    factor       = list(factors) if isinstance(factors, (list, tuple)) else [factors]
    factor1      = factor[0]
    counts       = df.groupby(factor1, observed=True).size()
    nbal         = counts.min()
    keymin       = counts.idxmin()
    factor1_list = counts.index
    if print_details:
        print('='*25)
        print(f"Key of {factor1} with minimum entries -> {keymin}")
        if len(factor) >= 2:
            factor2 = factor[1]
            details = df.loc[df[factor1] == keymin].groupby(factor2, observed=True).size()
            print(details)
        print('='*25)
    return int(nbal), factor1_list

def weighted_subsampling(df2subsample: pd.DataFrame, factors: Iterable[str], nbal: int, factor_val: str, rng_seed: Optional[int] = None) -> pd.DataFrame:
    """Subsample nbal rows from df where factor1 == chosen value,
    weighting by inverse counts of factor2 to balance within factor1."""
    factor = list(factors)
    factor1, factor2 = factor[0], factor[1]
    filtered = df2subsample.loc[df2subsample[factor1] == factor_val]
    if filtered.empty: raise ValueError(f"No rows found for {factor1} == {factor_val!r}")
    # ---- weights = 1 / count_per_factor2 (normalized)
    counts_per_f2 = filtered.groupby(factor2, observed=True)[factor1].transform("count")
    weights = (1.0 / counts_per_f2).astype(float)
    weights /= weights.sum()
    return filtered.sample(n=nbal, weights=weights, random_state=rng_seed)

def build_balanced_training_set(data2balance: pd.DataFrame,factors: Iterable[str], sampler_seed: Optional[int] = None) -> pd.DataFrame:
    """For each label of factor1, sample 'nbal' rows (weighted across factor2)."""
    nbal, factor1_list = find_minimum(data2balance, factors, print_details=False)
    print(f"\t\t\t\t\t[nbal] = {nbal}")
    # factor1 = list(factors)[0]
    parts = []
    for ft in factor1_list:
        sub = weighted_subsampling(df2subsample=data2balance, factors=factors, nbal=nbal, factor_val=ft, rng_seed=sampler_seed)
        parts.append(sub)
    df_balanced = pd.concat(parts, axis=0, ignore_index=False)
    return df_balanced

def testing_loop(mastersheet:  pd.DataFrame,
                 RFC: RandomForestClassifier,
                 factors:      Iterable[str],                 # e.g., ("KnownSpecies","EncounterNumber") or similar
                #  rfc_params:    Dict[str, Any],              # e.g., {"n_estimators":..., "max_features":..., "max_depth":..., "random_state": RS_rf, "oob_score": True}
                 sampler_seed: Optional[int] = None,     # seed used in subsampling
                 sep = '='*50
                ) -> Dict[str, Any]:
    """For each factor1 label:
            For each factor2 label under that factor1:
                - Train on all data EXCEPT rows with that factor2 label
                - Balance by factor1 using subsampling
                - Fit RandomForest
                - Predict on the held-out (factor1==label, factor2==held-out)
            Returns nested dict with predictions, probabilities, votes, etc."""
    factor1, factor2 = list(factors)[:2]
    factor1_list = mastersheet[factor1].drop_duplicates().reset_index(drop=True)
    results: Dict[str, Any] = {}
    print(f"\n{sep}\nIteration for each {factor1}:")
    # RFC = RandomForestClassifier(**rfc_params)
    for ft1 in factor1_list:
        print(f"\n\t\t{ft1}:")
        df_ft1 = mastersheet.loc[mastersheet[factor1] == ft1]
        ft2_ft1 = df_ft1[factor2].drop_duplicates().reset_index(drop=True)
        ft2_indexes, ft2_classification, ft2_tree_votes, ft2_class_prob, ft2_OOB_score = [], [], [], [], []
        class_params = None  # CHANGE: define before inner loop (scope clarity)
        
        print(f"\n\t\t\tIteration for each {factor2}:")
        for ft2 in ft2_ft1:
            print(f"\t\t\t\t{ft2}:")
            # print(f"\t{ft2}:\n\t\tBuilding balanced training set...")
            data2balance = mastersheet.loc[mastersheet[factor2] != ft2] # exclude this factor2 globally
            # ---- Balance training set across factor1
            df_balanced = build_balanced_training_set(data2balance=data2balance, factors=(factor1, factor2),sampler_seed=sampler_seed)
            X_train = df_balanced.drop(columns=[factor1, factor2])
            y_train = df_balanced[factor1]
            # ---- RFC fit
            # print('\t\tFitting model...')         
            RFC.fit(X_train, y_train)
            # ---- Build run set (held-out block)
            df2run = df_ft1.loc[df_ft1[factor2] == ft2]
            X_run  = df2run.drop(columns=[factor1,factor2])
            y_pred = RFC.predict(X_run)
            class_params = RFC.get_params(deep=False)         
            class_labels = RFC.classes_
            OOB_score    = getattr(RFC, "oob_score_", None)      
            class_prob     = RFC.predict_proba(X_run)
            class_prob_df  = pd.DataFrame(class_prob, columns=class_labels)
            total_trees    = RFC.n_estimators
            absolute_votes = (class_prob * total_trees)
            absolute_votes_class_round = pd.DataFrame(absolute_votes, columns=class_labels).round(3)
            # ---- Collect outputs
            ft2_indexes.append(df2run.index.values.tolist())
            ft2_classification.append(y_pred.tolist())
            ft2_class_prob.append(class_prob_df)
            ft2_tree_votes.append(absolute_votes_class_round)
            ft2_OOB_score.append(OOB_score)

        results[ft1] = {factor2: ft2_ft1.tolist(),"Indexes_raw": ft2_indexes,
            "Classification": ft2_classification, "Class_probs": ft2_class_prob,
            "Tree_votes": ft2_tree_votes, "Model_parameters": class_params,  "OOB_score": ft2_OOB_score}
    return results

def collect_true_pred_and_probs(
    results: Dict[str, Dict[str, Any]],
    species: Optional[Iterable[str]] = None,
    warn: bool = True,
    probs_suffix: Optional[str] = None
) -> Tuple[np.ndarray, np.ndarray, List[str], pd.DataFrame]:
    """
    Collects:
      - y_true, y_pred, classes (from 'Classification')
      - probs_df: concatenated 'Class_probs' with [UID, EncounterNumber, KnownSpecies]

    If probs_suffix is given, automatically suffixes all class-prob columns
    (i.e. the ones corresponding to species).
    """
    sp_order = list(species) if species is not None else list(results.keys())
    y_true_parts, y_pred_parts, prob_frames = [], [], []

    for sp in sp_order:
        if sp not in results:
            if warn: warnings.warn(f"[collect] Species '{sp}' not in results; skipping.", RuntimeWarning)
            continue

        d = results[sp]
        idx_lists  = d.get("Indexes_raw", [])
        groups     = d.get("EncounterNumber", [])
        yhat_ll    = d.get("Classification", [])
        probs_list = d.get("Class_probs", [])

        if not (idx_lists and groups and yhat_ll and probs_list):
            if warn: warnings.warn(f"[collect] Missing fields for '{sp}'; skipping.", RuntimeWarning)
            continue

        if not (len(idx_lists) == len(groups) == len(yhat_ll) == len(probs_list)):
            if warn: warnings.warn(f"[collect] Length mismatch for '{sp}'; skipping species.", RuntimeWarning)
            continue

        # Flatten predictions
        pred_sp, true_sp, sp_frames = [], [], []
        for idxs, grp, yhat, probs_df in zip(idx_lists, groups, yhat_ll, probs_list):
            if not (len(idxs) == len(probs_df) == len(yhat)):
                if warn: warnings.warn(f"[collect] Row mismatch for '{sp}' chunk; skipping chunk.", RuntimeWarning)
                continue
            pred_sp.extend(yhat)
            true_sp.extend([sp] * len(yhat))
            sp_frames.append(
                pd.concat(
                    [probs_df.reset_index(drop=True),
                     pd.DataFrame({"Species_Classification":yhat,"UID": idxs, "EncounterNumber": [grp]*len(idxs)})],
                    axis=1
                )
            )
        if not pred_sp or not sp_frames:
            continue

        y_true_parts.append(np.array(true_sp, dtype=object))
        y_pred_parts.append(np.array(pred_sp, dtype=object))
        df_sp = pd.concat(sp_frames, ignore_index=True)
        df_sp["KnownSpecies"] = sp
        prob_frames.append(df_sp)

    if not y_true_parts:
        raise ValueError("No predictions found to build y_true/y_pred.")
    y_true = np.concatenate(y_true_parts)
    y_pred = np.concatenate(y_pred_parts)

    # classes present, in requested order
    present = set(pd.unique(y_true))
    classes = [sp for sp in sp_order if sp in present]

    # probs table
    if not prob_frames:
        raise ValueError("No probability chunks found.")
    probs_df = pd.concat(prob_frames, ignore_index=True)

    # --- SMART SUFFIXING ---
    if probs_suffix:
        cols = probs_df.columns.tolist()
        # Identify which are class-prob columns:
        # use intersection between DataFrame columns and detected classes
        prob_cols = [c for c in cols if c in classes]
        # Suffix only those
        rename_map = {c: c + probs_suffix for c in prob_cols}
        probs_df = probs_df.rename(columns=rename_map)

    return y_true, y_pred, classes, probs_df

# --- LOGO EVENT FUNCTIONS ---
def summarize_encounter_votes(votes_df: pd.DataFrame, actual_species: str) -> dict:
    """
    Summarize one encounter's tree-vote matrix into top-1, top-2, scores, etc.
    Matches your original branching (ties, single-whistle encounters, etc.).
    """
    n_whistle = int(votes_df.shape[0])
    sum_tree = votes_df.sum(axis=0)
    total = float(sum_tree.sum())
    # avoid div-by-zero (unlikely if votes exist)
    enc_tree_prop = (sum_tree / total) if total > 0 else sum_tree * 0.0

    score_top = float(enc_tree_prop.max())
    winners_mask = (enc_tree_prop == score_top)
    winners = enc_tree_prop.index[winners_mask].tolist()

    # actual species score
    actual_species_score = float(enc_tree_prop.get(actual_species, 0.0))
    actual_species_score = round(actual_species_score, 2)

    if len(winners) == 1:
        species_classification = winners[0]
        # 2nd best only meaningful if >1 whistle
        if n_whistle > 1:
            drop_max = enc_tree_prop.drop(species_classification)
            score_2nd = float(drop_max.max())
            idx_2nd = drop_max == score_2nd
            second_list = drop_max.index[idx_2nd].tolist()
            species_2nd = second_list[0] if len(second_list) == 1 else " - ".join(second_list)
        else:
            score_2nd, species_2nd = None, None
    else:
        # tie in first place → replicate your logic (2nd == first, labels equal)
        species_classification = " - ".join(winners)
        score_2nd = float(score_top)
        species_2nd = species_classification

    return {
        "Score_classification": score_top,
        "Species_classification": species_classification,
        "Actual_species_score": actual_species_score,
        "Score_2nd": score_2nd,
        "Species_2nd": species_2nd,
        "N_whistle": n_whistle,
    }

def make_event_level_probs(probs_df: pd.DataFrame, encounter_col: str = "EncounterNumber",
                           meta_cols: tuple = ("Species_Classification", "UID", "EncounterNumber", "KnownSpecies"), suffix: str = "_LOGO_EVENT") -> pd.DataFrame:
    """Build event-level class proportions (sum over whistles per encounter, normalize)
    and repeat per-whistle. Auto-detect class-prob columns as those before meta cols."""
    cols = probs_df.columns.tolist()
    stop_idx = min([cols.index(c) for c in meta_cols if c in cols] or [len(cols)])
    class_cols = cols[:stop_idx]
    other_cols = cols[stop_idx:]
    if not class_cols:  raise ValueError("No class-probability columns detected.")
    out = []
    for enc, g in probs_df.groupby(encounter_col, sort=False):
        sums = g[class_cols].sum(axis=0)
        tot = float(sums.sum())
        props = (sums / tot) if tot > 0 else sums * 0.0
        # repeat per whistle in the encounter (preserve index to keep row order)
        repeated = pd.DataFrame([props.values] * len(g), columns=class_cols, index=g.index)
        out.append(pd.concat([repeated, g[other_cols]], axis=1))
    ev_tbl = pd.concat(out, ignore_index=False).reset_index(drop=True)
    # replace suffix with new suffix
    ev_tbl = ev_tbl.rename(columns={c: f"{c.split('_LOGO')[0]}{suffix}" if "_LOGO" in c else f"{c}{suffix}" 
                 for c in class_cols})
    return ev_tbl


Cetacea Whistle Classifier — Master Notebook
Starting code, imports, and global variables
	Python: 3.13.7
	NumPy: 2.3.2
	Pandas: 2.3.2
	Scikit-learn: 1.7.1


## Reading mastersheet

In [4]:
RUN_DIR = os.path.join(OUT_DIR, name_box.value)
FILE_DIR = os.path.join(DATA_DIR, file_dropdown.value)
os.makedirs(RUN_DIR, exist_ok=True)

factors = ("KnownSpecies", "EncounterNumber")  
MASTERSHEET = read_ROCCA_sheet(FILE_DIR, type='precleaned')
print(f'\nMaster sheet -> {FILE_DIR}\n')
MASTERSHEET.info()
# Count the number of events and whistles by species 
# Precompute totals once
total_rows   = len(MASTERSHEET)
total_events = MASTERSHEET[factors[1]].nunique()  # global unique encounters
# Single pass groupby with both metrics
by_sp = (MASTERSHEET.groupby(factors[0], observed=True).agg(
            n_events =(factors[1], "nunique"), # unique encounters per species
            n_whistle=(factors[1], "size"),    # rows per species
        ))

# Add percentages (0–100) and tidy
summary = (by_sp.assign(**{
                "% events":  (by_sp["n_events"]  / total_events * 100).round(2),
                "% whistle": (by_sp["n_whistle"] / total_rows   * 100).round(2),
            }).sort_values(["n_events", "n_whistle"], ascending=False))

# Simple headline stats
n_species    = summary.shape[0]
species_min = summary.n_whistle.idxmin()
print(f"\nTotal number of species: {n_species}")
print(f"Total number of events: {total_events}")
print("\nNumber of events/whistles by species:")
print(summary)
print(f"\nEncounter whistles of {species_min}:")
print(f"{MASTERSHEET.loc[MASTERSHEET[factors[0]] == species_min].groupby(factors[1], observed=True).size()}")

# Define the predictors and target from training balanced dataset
if 'UID' in MASTERSHEET.columns: X = MASTERSHEET.drop(['UID'], axis=1)
else: X = MASTERSHEET.copy()
# X.info()
Y = MASTERSHEET.loc[:, factors[0]]
Y.shape


Master sheet -> d:\Documentos\GITHUB\Cetacea_Classifier_AParo\inputs\Master_Whistle_7sp.csv

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 47 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   EncounterNumber       4622 non-null   object 
 1   KnownSpecies          4622 non-null   object 
 2   FREQMAX               4622 non-null   float64
 3   FREQMIN               4622 non-null   float64
 4   DURATION              4622 non-null   float64
 5   FREQBEG               4622 non-null   float64
 6   FREQEND               4622 non-null   float64
 7   FREQRANGE             4622 non-null   float64
 8   FREQMEAN              4622 non-null   float64
 9   FREQSTDDEV            4622 non-null   float64
 10  FREQMEDIAN            4622 non-null   float64
 11  FREQCENTER            4622 non-null   float64
 12  FREQRELBW             4622 non-null   float64
 13  FREQMAXMINRATIO       4622 no

(4622,)

## KFOLD
Edit the presets dictionary as prefer

In [ ]:
KFOLD_DIR = os.path.join(RUN_DIR, "KFOLD")
os.makedirs(KFOLD_DIR, exist_ok=True)
KFOLD_PRESETS = {
                "preset_labels":[   "A",    "B",    "C",   "D",   "E"], # Preset labels
                # Balancing Random states
                "rs_bal_1":     [ 56450,  97645,  63454,  9771, 50978], # 1st Random state for balancing methods (e.g. SMOTE/undersampling/oversampling)
                "rs_bal_2":     [  6273,  86067,   2116, 36711, 58443], # 2nd Random state for balancing methods (e.g. SMOTE/undersampling/oversampling)
                "rs_bal_3":     [  7833,  62224,  79859, 25882, 13185], # 3rd Random state for balancing methods (e.g. SMOTE/undersampling/oversampling)
                "rs_skf":       [ 31853,  76360,  95598, 82486, 23093], # Random state for stratified K-fold splitting
                # RandomForest hyperparameters
                "random_state": [ 13444,  92667,  34601, 51901, 25793], # Random state for the RandomForest classifier
                "n_estimators": [   200,    500,   1000,   200,  1000], # number of trees
                "max_features": ["log2", "log2", "log2",    10,  0.75], # feature selection strategy
                "max_depth":    [    20,     10,     15,    15,    15], # maximum tree depth
                    }

print(f"{sep}\nKFOLD Classification:\n{sep}")

# Iteration over each KFOLD preset 
presets_models_kfold = {} # to store the fit models for each preset
presets_probs_kfold  = [] # to store the probabilities for each preset
print('Iterating over each preset:')
for i, preset in enumerate(KFOLD_PRESETS['preset_labels']):
    print(f"\n{sep}----- Preset {preset} -----\n")
    cfg, rfc_params = get_preset_configs(KFOLD_PRESETS, KFOLD_DIR, preset, grouping_type='KFOLD')

    # Initialize the RandomForest model with the preset configuration
    print("\nInitializing Random Forest Classifier model...")  
    RFC = RandomForestClassifier(**rfc_params)
    SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state= cfg["rs_skf"]) # Create Stratified K-Fold object
    n_samples = len(X)

    # Iteration over each fold
    print('\nIterating over each fold...')
    oof_probabilities = np.zeros((X.shape[0], len(np.unique(Y)))) # Prepare to collect out-of-fold probabilities
    fold_models       = [] # to store the fit models for each fold
    for fold_idx, (train_index, test_index) in enumerate(SKF.split(X.drop(columns=['EncounterNumber', 'KnownSpecies']), Y)):
        print(f'Fold {fold_idx+1}:')
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

        # Calculate weights for balancing
        weights = {}
        for class_label in y_train.unique():
            # Use EncounterNumber for group calculations
            group_sizes = X_train[X_train['KnownSpecies'] == class_label].groupby('EncounterNumber').size()
            inverse_group_sizes  = 1 / group_sizes
            weights[class_label] = (inverse_group_sizes / inverse_group_sizes.sum()).to_dict()
        
        # Determine target class size and balance the training data within the fold        
        target_class_size = y_train.value_counts().min() 
        X_train_balanced, y_train_balanced = balance_train_data_kfold(X_train, weights, target_class_size,
                                                                      RS_bal_1= cfg["rs_bal_1"], RS_bal_2= cfg["rs_bal_2"], RS_bal_3= cfg["rs_bal_3"])
        n_train_bal_groups = len(X_train_balanced['EncounterNumber'].unique())
        # Remove 'EncounterNumber' and 'KnownSpecies' to train+fit the model and predict test 
        X_train_balanced = X_train_balanced.drop(columns=['EncounterNumber', 'KnownSpecies'], errors='ignore')
        X_test_features  = X_test.drop(columns=['EncounterNumber', 'KnownSpecies'], errors='ignore')
        RFC.fit(X_train_balanced, y_train_balanced) # Train the model on the balanced training fold
        fold_models.append(RFC)  # Store the trained model for this fold
        oof_probabilities[test_index] = RFC.predict_proba(X_test_features)

        if print_fold_stats:
            print('=' * 25)
            y_pred = RFC.predict(X_test_features)
            print("\t[info] Classification report:") # Detailed classification report for this fold
            class_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            print(f'\t\tOverall Accuracy:  {class_report['accuracy']}')
            print(f'\t\tBalanced Accuracy: {balanced_accuracy_score(y_test, y_pred)}')
            print(f'\t\tPrecision Macro:   {class_report['macro avg']['precision']}')
            print(f'\t\tRecall Macro:      {class_report['macro avg']['recall']}')
            print(f'\t\tF1 Macro:          {class_report['macro avg']['f1-score']}')
            print(f'\t\tF1 Weighted:       {class_report['weighted avg']['f1-score']}')
            print("\ntTraining set statistics:")
            total_train = len(X_train)
            print(f'\t\tTraining indexes: {train_index}')
            print(f'\t\tTotal training ocurrences: {total_train} ({total_train / n_samples:.2%})')
            print(f'\t\tTotal number of species: {len(y_train.unique())}')
            print(f'\t\tTotal number of events:  {len(X_train['EncounterNumber'].unique())}')
            print(y_train.value_counts())
            print("\n\tBalanced training set statistics:")
            total_train_bal = len(X_train_balanced)
            print(f'\t\tTotal training ocurrences: {total_train_bal} ({total_train_bal / n_samples:.2%})')
            print(f'\t\tTotal number of species: {len(y_train_balanced.unique())}')
            print(f'\t\tTotal number of events:  {n_train_bal_groups}')
            print(y_train_balanced.value_counts())
            print("\n\tTesting set statistics:")
            test_species_counts = y_test.value_counts()
            total_test = len(X_test)
            print(f'\t\tTotal testing ocurrences: {total_test} ({total_test / n_samples:.2%})')
            print(f'\t\tTotal number of species: {len(y_test.unique())}')
            print(f'\t\tTotal number of events:  {len(X_test['EncounterNumber'].unique())}')
            print(test_species_counts)
            labels_no_predictions = []
            for label, metrics in class_report.items(): # Find labels with no predictions
                if isinstance(metrics, dict) and metrics['precision'] == 0 and test_species_counts.get(label, 0) > 0:
                    labels_no_predictions.append(label)
            if labels_no_predictions:
                print(f"\t\tLabels with no predictions: {labels_no_predictions}")
            print('=' * 25)
    print('\nEnd of Fold iteration.\n','=' * 50)
    presets_models_kfold[preset] = fold_models # Store all models for this preset

    # Classification Out Of Folds (OOF) Probabilities and Weights
    # `oof_probabilities` [shape (n_samples, n_classes)] contains the predicted probabilities for all instances in the original dataset.
    class_labels = RFC.classes_      # Get the class labels from the trained model
    oof_df = pd.DataFrame(oof_probabilities, columns=class_labels)
    oof_df['Species_Classification'] = oof_df.idxmax(axis=1)
    if 'UID' in MASTERSHEET.columns:
        oof_df['UID']         = MASTERSHEET['UID'].values
    else: 
        oof_df['UID']         = MASTERSHEET.index.values
    oof_df['KnownSpecies']    = MASTERSHEET['KnownSpecies'].values  
    oof_df['EncounterNumber'] = MASTERSHEET['EncounterNumber'].values 
    oof_df.columns = [col + '_KFOLD_5' if i < n_species else col for i, col in enumerate(oof_df.columns)]
    print('\n[info] OOF table:')
    oof_df.info()
    # Saving OOF Probabilities table
    oof_probs_path = os.path.join(KFOLD_DIR, f'{preset}_KFOLD_OOF_Class_Probs.csv')
    print(f"\n[saving] OOF probabilities at {oof_probs_path}")
    oof_df.to_csv(oof_probs_path, index = False)
    presets_probs_kfold.append(oof_df)

    # Classifcation Report
    y_pred_oof  = oof_df['Species_Classification']
    y_test_oof  = oof_df['KnownSpecies']
    bal_acc_off = balanced_accuracy_score(y_test_oof, y_pred_oof)
    class_report_oof = classification_report(y_test_oof, y_pred_oof, output_dict=True, labels = class_labels )
    class_report_oof_df = pd.DataFrame(class_report_oof).transpose()
    class_report_oof_df['Labels'] = class_report_oof_df.index
    class_report_oof_df = class_report_oof_df[['Labels', 'precision', 'recall', 'f1-score', 'support']] # Reorder columns to have labels first
    # create new line with 'Labels' as 'balanced_accuracy' and 'precision' as the value of balanced_accuracy
    new_row = pd.DataFrame([['balanced_accuracy', bal_acc_off] + [None] * (class_report_oof_df.shape[1] - 2)],
                        columns=class_report_oof_df.columns)
    class_report_oof_df = pd.concat([class_report_oof_df, new_row], ignore_index=True)
    # Saving report
    report_path = os.path.join(KFOLD_DIR, f'{preset}_KFOLD_OOF_Class_Report.csv')
    print(f"\n[saving] OOF classification report at {report_path}")
    class_report_oof_df.to_csv(report_path, index = False)

    # Plotting Heatmap for the Confusion Matrix OFF \ WEIGHTS        
    conf_matrix_kfold = confusion_matrix(y_test_oof, y_pred_oof, labels = class_labels)
    conf_matrix_adjusted_kfold = adjust_confusion_matrix(conf_matrix_kfold)
    plot_confusion_matrix_heatmap(matrix=conf_matrix_adjusted_kfold, class_labels=class_labels,
            preset=preset,save_dir=KFOLD_DIR,grouping="KFOLD_OOF")
    
print('\nEnd of presets iteration.\n','=' * 50)
averaged_probs_kfold = average_presets_probs(tables=presets_probs_kfold,
    presets_labels=KFOLD_PRESETS['preset_labels'], out_dir=KFOLD_DIR, method_tag="KFOLD")
print('=' * 50,'\nEnd of KFOLD Classification.\n','=' * 50)


## Leave One Group Out (LOGO)
-Individual (Whistles) and Events (Encounter) level classification

In [36]:
LOGO_DIR = os.path.join(RUN_DIR, "LOGO")
os.makedirs(LOGO_DIR, exist_ok=True)
LOGO_PRESETS = {
                "preset_labels": [   "A",   "B",    "C",   "D",    "E"], # Preset labels
                "rs_bal":        [ 26225, 64386,  61999, 71437,   1972], # Random state for balancing methods (e.g. SMOTE/undersampling/oversampling)
                # RandomForest hyperparameters
                "random_state":  [ 78978, 52029,  78121, 78113,  72661], # Random state for the RandomForest classifier
                "n_estimators":  [   500,  1000,   1000,   500,   1000], # number of trees
                "max_features":  ["log2",   0.5, "log2",   0.5, 'sqrt'], # feature selection strategy
                "max_depth":     [    10,    10,     30,    10,     10], # maximum tree depth
                    }

presets_models_logo_ind = {} # to store the fit models for each preset (individual)
presets_models_logo_ev  = {} # to store the fit models for each preset (event-level)
presets_probs_logo_ind  = [] # to store the probabilities for each preset (individual)
presets_probs_logo_ev   = [] # to store the probabilities for each preset (event-level)

print(f'{sep}\n  Leave One Group Out (LOGO) Classification:\n{sep}')
print('Iterating over each preset:')
for i, preset in enumerate(LOGO_PRESETS['preset_labels']):
    print(f"{sep}\n---------- Preset {preset} ----------")
    cfg, rfc_params = get_preset_configs(LOGO_PRESETS, LOGO_DIR, preset, grouping_type='LOGO')
    print("\nInitializing Random Forest Classifier model...")
    RFC = RandomForestClassifier(**rfc_params)
    preset_results = testing_loop(mastersheet=X,RFC=RFC, factors=factors, sampler_seed=cfg["rs_bal"])

    # --- Probabilities table and classifier results ---
    y_true, y_pred, class_labels, probs_individual = collect_true_pred_and_probs(preset_results, probs_suffix="_LOGO_IND")
    print(f"[info] Individual class-prob table shape: {probs_individual.shape}")
    probs_individual.info()
    # --- Saving probabilities table ---
    probs_ind_path = os.path.join(LOGO_DIR, f"{preset}_LOGO_INDIVIDUAL_Class_Probs.csv")
    probs_individual.to_csv(probs_ind_path, index=False)
    print(f"[saving] {probs_ind_path}")
    presets_probs_logo_ind.append(probs_individual)

    # --- Classification report ---
    report_dict = classification_report(y_true, y_pred, output_dict=True, labels=class_labels, zero_division=0)
    report_df   = (pd.DataFrame(report_dict).transpose().rename_axis("Labels").reset_index())
    cols = ["Labels", "precision", "recall", "f1-score", "support"] # Keep only useful cols
    class_report_individual = report_df[[c for c in cols if c in report_df.columns]]
    # class_report_individual = class_report_individual[cols]
    balanced_acc = balanced_accuracy_score(y_true, y_pred)  # Balanced accuracy row
    new_row = pd.DataFrame([["balanced_accuracy", balanced_acc, None, None, None]],columns=class_report_individual.columns)
    class_report_individual = pd.concat([class_report_individual, new_row], ignore_index=True)
    print('[info] LOGO Individual Classification Report:')
    print(class_report_individual)
    report_ind_path = os.path.join(LOGO_DIR, f'{preset}_LOGO_INDIVIDUAL_Class_Report.csv')
    print(f"\n[saving] {report_ind_path}")
    class_report_individual.to_csv(report_ind_path, index = False)

    # --- Confusion Matrix plot ---
    conf_matrix = confusion_matrix(y_true, y_pred, labels=class_labels)
    conf_matrix_adjusted = adjust_confusion_matrix(conf_matrix)
    plot_confusion_matrix_heatmap(matrix=conf_matrix_adjusted, class_labels=class_labels,
                preset=preset,save_dir=LOGO_DIR,grouping="LOGO_INDIVIDUAL")
    
    # --- LOGO Events Classification ---
    print('='*50,'\nLOGO Events Classification:\n')
    # --- Events Probabilities table ---
    probs_events = make_event_level_probs(probs_individual, suffix="_LOGO_EVENT")
    print(f"[info] Events Probabilities table shape: {probs_events.shape}")
    probs_events.info()
    probs_evts_path = os.path.join(LOGO_DIR, f"{preset}_LOGO_EVENTS_Class_Probs.csv")
    print(f"\n[saving] {probs_evts_path}")
    probs_events.to_csv(probs_evts_path, index=False)
    presets_probs_logo_ev.append(probs_events)

    # ---- Summarize each encounter from Tree_votes ----
    OOB_score_list, enc_summary_frames = [], []
    for species, res in preset_results.items():
        # gather OOB scores
        OOB_score_list.extend(res.get("OOB_score", []))
        tree_votes_list = res["Tree_votes"]           # list[pd.DataFrame]
        encounter_ids   = res["EncounterNumber"]      # list[str], same length
        summaries = []
        for votes_df, enc_id in zip(tree_votes_list, encounter_ids):
            s = summarize_encounter_votes(votes_df, actual_species=species)
            s["KnownSpecies"] = species
            s["EncounterNumber"] = enc_id
            summaries.append(s)
        df_sp = pd.DataFrame(summaries)
        df_sp["Correct"] = (df_sp["Species_classification"] == df_sp["KnownSpecies"])
        enc_summary_frames.append(df_sp)
    Enc_Class_results = pd.concat(enc_summary_frames, ignore_index=True) # Concatenate all species

    # ---- Event Classification Report and Metrics ----
    y_test_ev = Enc_Class_results["KnownSpecies"]
    y_pred_ev = Enc_Class_results["Species_classification"]
    class_ev  = Enc_Class_results["KnownSpecies"].unique()
    class_report_events = classification_report(y_test_ev, y_pred_ev, output_dict=True, labels=class_ev)
    class_report_events = (pd.DataFrame(class_report_events).transpose().rename_axis("Labels")
                            .reset_index()[["Labels", "precision", "recall", "f1-score", "support"]])
    mean_OOB = float(np.mean(OOB_score_list)) if OOB_score_list else np.nan
    sd_OOB   = float(np.std(OOB_score_list))  if OOB_score_list else np.nan
    accuracy = 100.0 * accuracy_score(y_test_ev, y_pred_ev)
    balanced_acc = balanced_accuracy_score(y_test_ev, y_pred_ev)
    new_row = pd.DataFrame([["balanced_accuracy", balanced_acc, None, None, None]],
                        columns=class_report_events.columns)
    class_report_events = pd.concat([class_report_events, new_row], ignore_index=True) # add balanced_accuracy row
    print('\n[info] Events Classification Report:')
    print(f"\tEvent Accuracy: {accuracy:.2f}%")
    print(f"\tMean OOB: {mean_OOB:.2f}")
    print(f"\tSD OOB: {sd_OOB:.2f}\n")
    print(class_report_events)
    report_events_path = os.path.join(LOGO_DIR, f'{preset}_LOGO_EVENTS_Class_Report.csv')
    print(f"\n[saving] {report_events_path}")
    class_report_events.to_csv(report_events_path, index = False)

    # --- Confusion Matrix plot ---
    conf_matrix_ev = confusion_matrix(y_test_ev, y_pred_ev, labels=class_ev)
    conf_matrix_ev_adjusted = adjust_confusion_matrix(conf_matrix_ev) 
    plot_confusion_matrix_heatmap(matrix=conf_matrix_ev_adjusted, class_labels=class_labels,
                preset=preset,save_dir=LOGO_DIR,grouping="LOGO_EVENTS")

print('\nEnd of presets iteration.\n','=' * 50)
averaged_probs_logo_ind = average_presets_probs(tables=presets_probs_logo_ind,
    presets_labels=LOGO_PRESETS['preset_labels'], out_dir=LOGO_DIR, method_tag="LOGO_INDIVIDUAL")
averaged_probs_logo_ev = average_presets_probs(tables=presets_probs_logo_ev,
    presets_labels=LOGO_PRESETS['preset_labels'], out_dir=LOGO_DIR, method_tag="LOGO_EVENT")
print('=' * 50,'\nEnd of LOGO Classification.\n','=' * 50)


  Leave One Group Out (LOGO) Classification:
Iterating over each preset:
---------- Preset A ----------
Configuration:
	      rs_bal: 26225
	random_state: 78978
	n_estimators: 500
	max_features: log2
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_2_A14:
					[nbal] = 139
				PMC_4_A32:
					[nbal] = 139
				PMC_8_A101

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\2977092853.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_individual = pd.concat([class_report_individual, new_row], ignore_index=True)


LOGO Events Classification:

[info] Events Probabilities table shape: (4622, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Delphinus_LOGO_EVENT       4622 non-null   float64
 1   S_attenuata_LOGO_EVENT     4622 non-null   float64
 2   S_bredanensis_LOGO_EVENT   4622 non-null   float64
 3   S_clymene_LOGO_EVENT       4622 non-null   float64
 4   S_frontalis_LOGO_EVENT     4622 non-null   float64
 5   S_longirostris_LOGO_EVENT  4622 non-null   float64
 6   T_truncatus_LOGO_EVENT     4622 non-null   float64
 7   Species_Classification     4622 non-null   object 
 8   UID                        4622 non-null   int64  
 9   EncounterNumber            4622 non-null   object 
 10  KnownSpecies               4622 non-null   object 
dtypes: float64(7), int64(1), object(3)
memory usage: 397.3+ KB

[saving] d:\Documen

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\2977092853.py:100: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_events = pd.concat([class_report_events, new_row], ignore_index=True) # add balanced_accuracy row



[saving] Confusion Matrix LOGO_EVENTS at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\A_LOGO_EVENTS_Confusion_Matrix.png
---------- Preset B ----------
Configuration:
	      rs_bal: 64386
	random_state: 52029
	n_estimators: 1000
	max_features: 0.5
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_

...


[saving] Confusion Matrix LOGO_EVENTS at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\D_LOGO_EVENTS_Confusion_Matrix.png
---------- Preset E ----------
Configuration:
	      rs_bal: 1972
	random_state: 72661
	n_estimators: 1000
	max_features: sqrt
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\2977092853.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_individual = pd.concat([class_report_individual, new_row], ignore_index=True)



[saving] Confusion Matrix LOGO_INDIVIDUAL at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\E_LOGO_INDIVIDUAL_Confusion_Matrix.png
LOGO Events Classification:

[info] Events Probabilities table shape: (4622, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Delphinus_LOGO_EVENT       4622 non-null   float64
 1   S_attenuata_LOGO_EVENT     4622 non-null   float64
 2   S_bredanensis_LOGO_EVENT   4622 non-null   float64
 3   S_clymene_LOGO_EVENT       4622 non-null   float64
 4   S_frontalis_LOGO_EVENT     4622 non-null   float64
 5   S_longirostris_LOGO_EVENT  4622 non-null   float64
 6   T_truncatus_LOGO_EVENT     4622 non-null   float64
 7   Species_Classification     4622 non-null   object 
 8   UID                        4622 non-null   int64  
 9   EncounterNumber            4622 non-nu

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\2977092853.py:100: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_events = pd.concat([class_report_events, new_row], ignore_index=True) # add balanced_accuracy row



[saving] Confusion Matrix LOGO_EVENTS at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\E_LOGO_EVENTS_Confusion_Matrix.png

End of presets iteration.

Averaging probabilities...
[saving] d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\_ABCDE_LOGO_INDIVIDUAL_Averaged_Probs.csv

Averaging probabilities...
[saving] d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\_ABCDE_LOGO_EVENT_Averaged_Probs.csv
End of LOGO Classification.


  Leave One Group Out (LOGO) Classification:
Iterating over each preset:
---------- Preset A ----------
Configuration:
	      rs_bal: 26225
	random_state: 78978
	n_estimators: 500
	max_features: log2
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_2_A14:
					[nbal] = 139
				PMC_4_A32:
					[nbal] = 139
				PMC_8_A101

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\3888215221.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_individual = pd.concat([class_report_individual, new_row], ignore_index=True)



[saving] Confusion Matrix LOGO_INDIVIDUAL at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\A_LOGO_INDIVIDUAL_Confusion_Matrix.png
LOGO Events Classification:

[info] Events Probabilities table shape: (4622, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Delphinus_LOGO_EVENT       4622 non-null   float64
 1   S_attenuata_LOGO_EVENT     4622 non-null   float64
 2   S_bredanensis_LOGO_EVENT   4622 non-null   float64
 3   S_clymene_LOGO_EVENT       4622 non-null   float64
 4   S_frontalis_LOGO_EVENT     4622 non-null   float64
 5   S_longirostris_LOGO_EVENT  4622 non-null   float64
 6   T_truncatus_LOGO_EVENT     4622 non-null   float64
 7   Species_Classification     4622 non-null   object 
 8   UID                        4622 non-null   int64  
 9   EncounterNumber            4622 non-nu

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\3888215221.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_events = pd.concat([class_report_events, new_row], ignore_index=True) # add balanced_accuracy row



[saving] Confusion Matrix LOGO_EVENTS at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\A_LOGO_EVENTS_Confusion_Matrix.png
---------- Preset B ----------
Configuration:
	      rs_bal: 64386
	random_state: 52029
	n_estimators: 1000
	max_features: 0.5
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_

...


[saving] Confusion Matrix LOGO_EVENTS at d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\D_LOGO_EVENTS_Confusion_Matrix.png
---------- Preset E ----------
Configuration:
	      rs_bal: 1972
	random_state: 72661
	n_estimators: 1000
	max_features: sqrt
	   max_depth: 10
	   oob_score: True
	      n_jobs: -1

Initializing Random Forest Classifier model...

Iteration for each KnownSpecies:

		Delphinus:

			Iteration for each EncounterNumber:
				PMC_12_A15:
					[nbal] = 139
				PMC_12_A33:
					[nbal] = 139
				PMC_12_A46:
					[nbal] = 139
				PMC_12_A47:
					[nbal] = 139
				PMC_12_A62:
					[nbal] = 139
				PMC_3_A68:
					[nbal] = 139
				PMC_6_A51:
					[nbal] = 139
				PMC_9_A102:
					[nbal] = 139

		S_attenuata:

			Iteration for each EncounterNumber:
				Per_02_A05:
					[nbal] = 139
				Per_02_A23:
					[nbal] = 139
				Per_02_A56:
					[nbal] = 139
				PMC_12_A42:
					[nbal] = 139
				PMC_1_A13:
					[nbal] = 139
				PMC_1_A48:
					[nbal] = 139
				PMC_

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\3888215221.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_individual = pd.concat([class_report_individual, new_row], ignore_index=True)


LOGO Events Classification:

[info] Events Probabilities table shape: (4622, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Delphinus_LOGO_EVENT       4622 non-null   float64
 1   S_attenuata_LOGO_EVENT     4622 non-null   float64
 2   S_bredanensis_LOGO_EVENT   4622 non-null   float64
 3   S_clymene_LOGO_EVENT       4622 non-null   float64
 4   S_frontalis_LOGO_EVENT     4622 non-null   float64
 5   S_longirostris_LOGO_EVENT  4622 non-null   float64
 6   T_truncatus_LOGO_EVENT     4622 non-null   float64
 7   Species_Classification     4622 non-null   object 
 8   UID                        4622 non-null   int64  
 9   EncounterNumber            4622 non-null   object 
 10  KnownSpecies               4622 non-null   object 
dtypes: float64(7), int64(1), object(3)
memory usage: 397.3+ KB

[saving] d:\Documen

C:\Users\rafa_\AppData\Local\Temp\ipykernel_9856\3888215221.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_report_events = pd.concat([class_report_events, new_row], ignore_index=True) # add balanced_accuracy row



[info] Events Classification Report:
	Event Accuracy: 75.37%
	Mean OOB: 0.57
	SD OOB: 0.01

               Labels  precision    recall  f1-score     support
0           Delphinus   0.500000  0.750000  0.600000    8.000000
1         S_attenuata   0.529412  0.947368  0.679245   19.000000
2       S_bredanensis   0.600000  1.000000  0.750000    6.000000
3           S_clymene   0.857143  0.857143  0.857143   14.000000
4         S_frontalis   0.872340  0.863158  0.867725   95.000000
5      S_longirostris   0.714286  0.833333  0.769231   24.000000
6         T_truncatus   0.818182  0.243243  0.375000   37.000000
7            accuracy   0.753695  0.753695  0.753695    0.753695
8           macro avg   0.698766  0.784892  0.699763  203.000000
9        weighted avg   0.787915  0.753695  0.733872  203.000000
10  balanced_accuracy   0.784892       NaN       NaN         NaN

[saving] d:\Documentos\GITHUB\Cetacea_Classifier_AParo\outputs\baseline\LOGO\E_LOGO_EVENTS_Class_Report.csv

[saving] Confusio

## META Classifier (KFOLD + LOGO event + LOGO individual)

In [ ]:
META_DIR = os.path.join(RUN_DIR, "META")
os.makedirs(META_DIR, exist_ok=True)
META_PRESETS = {
                "preset_labels": [   "A",     "B",    "C",    "D",    "E"], # Preset labels
                # Random states
                "rs_bal":        [ 85139,   96070,  18189,  18133,  94916], # Random state for balancing methods (e.g. SMOTE/undersampling/oversampling)
                "rs_rf":         [ 50057,   35678,  39945,  34400,  73897], # Random state for the RandomForest classifier
                # RandomForest hyperparameters
                "n_estimators":  [   500,     100,   1000,    500,    100], # number of trees
                "max_features":  ["log2",  "log2", "log2", 'log2', 'log2'], # feature selection strategy
                "max_depth":     [    10,      10,     10,     10,     20], # maximum tree depth
                    }



In [ ]:
############## 
# AVG_PROB
##############

#%% CONCATENATE AVG PROBS AND CREATE META-FEATURES SHEET 

meta_feat_path = r'D:\WHISTLE_CLASSIFIER_FINAL\8sp\Meta_features_8sp'

meta_feat_name = '_8sp'

#path for class probs files
os.chdir(meta_feat_path)

os.listdir()

# Kf_5 = pd.read_csv('CLASS_PROBS_AVG_KFOLD_5.csv')
# Lg_wh = pd.read_csv('CLASS_PROBS_AVG_LOGO_WH.csv')
# Lg_we = pd.read_csv('CLASS_PROBS_AVG_LOGO_WE.csv')



# Collect all UIDs from the DataFrames
uids_Kf_5 = set(Kfold_mf['UID'])
uids_Lg_wh = set(logo_WH_mf['UID'])
uids_Lg_we = set(logo_WE_mf['UID'])

# Find UIDs that are unique to each DataFrame
unique_uids_Kf_5 = uids_Kf_5 - uids_Lg_wh - uids_Lg_we
unique_uids_Lg_wh = uids_Lg_wh - uids_Kf_5 - uids_Lg_we
unique_uids_Lg_we = uids_Lg_we - uids_Kf_5 - uids_Lg_wh

# Warn if there are UIDs not matching in all DataFrames
if unique_uids_Kf_5 or unique_uids_Lg_wh or unique_uids_Lg_we:
    print("Warning: Some UIDs do not match across DataFrames!")
    if unique_uids_Kf_5:
        print(f"UIDs unique to Kf_5: {unique_uids_Kf_5}")
    if unique_uids_Lg_wh:
        print(f"UIDs unique to Lg_wh: {unique_uids_Lg_wh}")
    if unique_uids_Lg_we:
        print(f"UIDs unique to Lg_we: {unique_uids_Lg_we}")
else:
    
    print('\n\t UIDs present in all files (no unique UIDs)')
    
    
# Merging the DataFrames on 'UID'
merged_df = Kfold_mf.merge(logo_WH_mf, on='UID', how='outer').merge(logo_WE_mf, on='UID', how='outer')

# Display the resulting DataFrame
print(merged_df)

###drop columns
merged_df.columns
columns_to_drop  = ['Species_Classification', 'KnownSpecies_x', 'EncounterNumber_x','KnownSpecies_y', 'EncounterNumber_y' ]

meta_features = merged_df.drop(columns = columns_to_drop)

#move UID to the last column

col_to_move = 'UID'
cols = list(meta_features.columns)

cols.remove(col_to_move)
cols.append(col_to_move)

meta_features = meta_features[cols]

meta_features.info()
meta_features.head()


# Save the merged DataFrame to a CSV file
meta_features.to_csv(f'META_FEATURES{meta_feat_name}.csv', index=False)


## Predict New Data

In [ ]:
# SELECT FOLDER WITH NEW DATA

In [ ]:
## DOUBLE CHECK this part from the predict new data code

# ==== Paths to your pre-balanced LOGO training data ====
csv_paths = [
    'ALL_DATA_bal_LOGO_7sp_A.csv',
    'ALL_DATA_bal_LOGO_7sp_B.csv',
    'ALL_DATA_bal_LOGO_7sp_C.csv',
    'ALL_DATA_bal_LOGO_7sp_D.csv',
    'ALL_DATA_bal_LOGO_7sp_E.csv'
]

def train_logo_ind_models(csv_paths, n_estimators_list, max_feat_list, max_depth_list, random_state_list):
    models_logo = {}

    for i in range(5):
        df = pd.read_csv(csv_paths[i])
        X_train = df.loc[:, 'FREQMAX':'STEPDUR']
        y_train = df['KnownSpecies']

        model = RandomForestClassifier(
            n_estimators=n_estimators_list[i],
            max_features=max_feat_list[i],
            max_depth=max_depth_list[i],
            random_state=random_state_list[i]
        )
        model.fit(X_train, y_train)
        models_logo[f'rf_model_logo_{chr(65 + i)}'] = model

    return models_logo


all_models_logo = train_logo_ind_models(csv_paths, n_estimators_list, max_feat_list, max_depth_list, random_state_list)

#%% FUNCTION TO PREDICT NEW DATA WITH THE LOGO MODELS
def predict_logo_ind_probs(wh_new_data_clean, all_models_logo):
    X_external = wh_new_data_clean.drop(columns=['KnownSpecies', 'EncounterNumber'], errors='ignore')

    logo_prob_list = []

    for model in all_models_logo.values():
        probas = model.predict_proba(X_external)
        class_names = model.classes_
        proba_df = pd.DataFrame(probas, columns=class_names, index=X_external.index)
        logo_prob_list.append(proba_df)

    avg_probs_logo_ind = sum(logo_prob_list) / len(logo_prob_list)
    avg_probs_logo_ind.columns = [f"{col}_LOGO_WH" for col in avg_probs_logo_ind.columns]
    
    return avg_probs_logo_ind, logo_prob_list, class_names


def compute_logo_group_probs(logo_prob_list, n_estimators_list, class_names, target_index):
    """
    Computes group-level averaged probabilities using model votes.
    """
   
    group_level_prob_list = []

    for group_df, n_estimators in zip(logo_prob_list, n_estimators_list):
        if group_df.empty:
            print("Warning: empty group_df encountered, skipping.")
            continue

        vote_matrix = group_df * n_estimators
        sum_votes = vote_matrix.sum(axis=0)
        total_votes = sum_votes.sum()

        if total_votes == 0:
            print("Warning: total votes = 0, setting all probs to 0")
            group_probs = pd.Series(0.0, index=class_names)
        else:
            group_probs = sum_votes / total_votes
            group_probs = group_probs.reindex(class_names, fill_value=0.0)

        group_level_prob_list.append(group_probs)

    avg_logo_group_probs = sum(group_level_prob_list) / len(group_level_prob_list)

    logo_group_df = pd.DataFrame(
        np.repeat([avg_logo_group_probs.values], len(target_index), axis=0),
        columns=[f"{cls}_LOGO_WE" for cls in avg_logo_group_probs.index],
        index=target_index
    )

    return logo_group_df, avg_logo_group_probs


probs_logo_group, avg_logo_group_probs  = compute_logo_group_probs(logo_prob_list, n_estimators_list, class_names, wh_new_data_clean.index)



In [ ]:
csv_paths = [
    'ALL_DATA_bal_LOGO_META_7sp_A.csv',
    'ALL_DATA_bal_LOGO_META_7sp_B.csv',
    'ALL_DATA_bal_LOGO_META_7sp_C.csv',
    'ALL_DATA_bal_LOGO_META_7sp_D.csv',
    'ALL_DATA_bal_LOGO_META_7sp_E.csv'
]

# ==== Model hyperparameters for each ====
n_estimators_list = [500, 100, 1000, 500, 100]
max_feat_list = ['log2', 'log2', 'log2', 'log2','log2']
max_depth_list = [10, 10, 10, 10, 20]
random_state_list = [50057, 35678, 39945, 34400, 73897]


def train_logo_meta_models(csv_paths, n_estimators_list, max_feat_list, max_depth_list, random_state_list):
    """
    Train LOGO meta-classifier models from provided pre-balanced CSVs.
    Returns a dictionary of trained models.
    """
    all_models_meta = {}

    for i in range(len(csv_paths)):
        df = pd.read_csv(csv_paths[i])
        X_train = df.loc[:, 'Delphinus_KFOLD_5':'T_truncatus_LOGO_WE']
        y_train = df['KnownSpecies']

        model = RandomForestClassifier(
            n_estimators=n_estimators_list[i],
            
            max_depth=max_depth_list[i],
            random_state=random_state_list[i]
        )

        model.fit(X_train, y_train)
        all_models_meta[f'rf_model_logo_{chr(65 + i)}'] = model

    return all_models_meta


##############################################################################
# TRAIN LOGO META MODELS 
##############################################################################


all_models_meta = train_logo_meta_models(csv_paths, n_estimators_list, max_feat_list, max_depth_list, random_state_list)

def predict_logo_meta_group_probs(meta_features_df, all_models_meta, n_estimators_list, class_names):
    """
    Predict group-level class probabilities using trained meta-classifier models.
    
    Parameters:
        meta_features_df: DataFrame with features (same columns used during training).
        all_models_meta: Dict of trained meta models.
        n_estimators_list: List of n_estimators for each model (for weighting votes).
        class_names: List of all class labels (same order as in training).
    
    Returns:
        avg_logo_group_meta_probs: Series with average group-level probabilities.
    """
    import pandas as pd

    logo_meta_prob_list = []

    for model in all_models_meta.values():
        probas = model.predict_proba(meta_features_df)
        proba_df = pd.DataFrame(probas, columns=model.classes_, index=meta_features_df.index)
        logo_meta_prob_list.append(proba_df)

    # Compute weighted vote aggregation
    group_level_meta_prob_list = []

    for group_df, n_estimators in zip(logo_meta_prob_list, n_estimators_list):
        if group_df.empty:
            print("Warning: empty group_df encountered, skipping.")
            continue

        vote_matrix = group_df * n_estimators
        sum_votes = vote_matrix.sum(axis=0)
        total_votes = sum_votes.sum()

        if total_votes == 0:
            print("Warning: total votes = 0, setting all probs to 0")
            group_probs = pd.Series(0.0, index=class_names)
        else:
            group_probs = sum_votes / total_votes
            group_probs = group_probs.reindex(class_names, fill_value=0.0)

        group_level_meta_prob_list.append(group_probs)

    avg_logo_group_meta_probs = sum(group_level_meta_prob_list) / len(group_level_meta_prob_list)
    avg_logo_group_meta_probs.name = "Avg_meta_group_probs"

    return avg_logo_group_meta_probs


In [ ]:

# Path to folder with new group CSVs
folder_path = r'C:\Users\alebi\Desktop\Code_for Whistle_predction\new_data_3'

results_output_folder = "Classification_results"
os.makedirs(results_output_folder, exist_ok=True)

# filename= 'S_attenuata_PMC_18_A29_RoccaContourStats.csv'
# Store predictions
all_group_predictions = []

logo_probs = []

meta_probs = []

# Loop through all CSV files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        
        print(f"\n\tProcessing Acoustic Event: {filename}")
        
        # Load new group data
        wh_new_data = pd.read_csv(filename)

        col_exclude = (

                      list(wh_new_data.loc[:, 'Source':'EncounterCount'].columns) +
                      list(wh_new_data.loc[:, 'SamplingRate':'DataProvider'].columns) + 
                                      ['GeographicLocation','ClassifiedSpecies', 'DCMEAN', 'DCSTDDEV'] +
                      list(wh_new_data.loc[:, 'DCQUARTER1MEAN':'DCQUARTER4MEAN'].columns) +
                      list(wh_new_data.loc[:, 'FREQBEGSWEEP':'FREQENDDWN'].columns) +
                      list(wh_new_data.loc[:, 'FREQPEAK':'Species_List'].columns)                                                         
                           
                      )

#clean columns 
        wh_new_data_clean = wh_new_data.drop(columns = col_exclude)


        # ======================
        # BASE CLASSIFIER 1: K-FOLD
        
        avg_probs_kfold = predict_kfold_probs(wh_new_data_clean, all_models_kfold)
        
        # BASE CLASSIFIER 2: LOGO individual
        avg_probs_logo_ind, logo_prob_list, class_names = predict_logo_ind_probs(wh_new_data_clean, all_models_logo)

        # BASE CLASSIFIER 3: LOGO group
        probs_logo_group, avg_logo_group_probs  = compute_logo_group_probs(logo_prob_list, n_estimators_list, class_names, wh_new_data_clean.index)


        # ======================
        # Build meta-features
        meta_features_df = pd.concat(
            [avg_probs_kfold, avg_probs_logo_ind, probs_logo_group],
            axis=1
        )

        #meta_features_df.info()

        #The assert statement in Python is used to test whether a condition is True.
        # If it’s True, nothing happens and the program continues. If it’s False, Python raises an AssertionError and stops execution
        assert (avg_probs_kfold.index == avg_probs_logo_ind.index).all()
        assert (avg_probs_kfold.index == probs_logo_group.index).all()


        # ======================
        # Predict with meta-classifier
        avg_logo_group_meta_probs = predict_logo_meta_group_probs(meta_features_df, all_models_meta, n_estimators_list, class_names)
        

        
        # Save the Kfoldprobs 
            
        avg_probs_kfold['Max_Score'] = avg_probs_kfold.max(axis=1)
        avg_probs_kfold['Classification'] = avg_probs_kfold.idxmax(axis=1)
        avg_probs_kfold.to_csv(os.path.join(results_output_folder, f'{filename}_Kfold.csv'))


        # Save the LOGO ind probs 
            
        avg_probs_logo_ind['Max_Score'] = avg_probs_logo_ind.max(axis=1)
        avg_probs_logo_ind['Classification'] = avg_probs_logo_ind.idxmax(axis=1)
        avg_probs_logo_ind.to_csv(os.path.join(results_output_folder, f'{filename}_LOGO_ind.csv'))







        # ======================
        # Classification Decisions

        # LOGO group
        max_score_logo = avg_logo_group_probs.max()
        max_index_logo = avg_logo_group_probs.idxmax()

        # META
        max_score_meta = avg_logo_group_meta_probs.max()
        max_index_meta = avg_logo_group_meta_probs.idxmax()

        # Final decision logic
        if max_index_logo == max_index_meta:
            final_class = max_index_meta
        elif max_index_meta == 'T_truncatus':
            final_class = max_index_meta
        else:
            final_class = max_index_logo

        # ======================
        # Print summary
        print('-------------------------------------------------------------------------------------------')
        species = wh_new_data_clean['KnownSpecies'].unique()[0]
    
        print(f'\nFile: {filename}')
        print(f'\nNumber of whistles: {len(wh_new_data)}\n')


        #print(avg_logo_group_probs)
        
        print(f"Species classification for the LOGO Classifier: \n\n\t {max_index_logo}\n\t Score: {round(max_score_logo, 2)}\n")

        #print(avg_logo_group_meta_probs)
      
        print(f"Species classifcation for the Meta-Classifier: \n\n\t {max_index_meta}\n\t Score: {round(max_score_meta, 2)}\n")
        print(f'\t\nTrue Species: {species}')
        print(f"Final Classification: {final_class}")
        
        print('-------------------------------------------------------------------------------------------')
        # ======================
        # Save results
        result_row = {
            'Species': species,
            "File": filename,
            "N_whistles": len(wh_new_data),
            "LOGO_Score": round(max_score_logo, 2),
            "LOGO_Classification": max_index_logo,
            "Meta_Score": round(max_score_meta, 2),
            "Meta_Classification": max_index_meta,
            "Final_Classification": final_class
                      }        
        
        
        all_group_predictions.append(result_row)
        
        
        logo_dict = {
        'Species': species,
        'File': filename,
        'N_whistles': len(wh_new_data),
        **avg_logo_group_probs.to_dict()# ** dictionary unpacking key-value to combine to other keys
                    }      
        logo_probs.append(logo_dict)
        
        
        meta_dict = {          
            'Species': species,
            'File': filename,
            'N_whistles': len(wh_new_data),
            **avg_logo_group_meta_probs.to_dict()
                    }
        
        meta_probs.append(meta_dict)
        
print(f'\nTask finished. {len(os.listdir(folder_path))-1} Acoustic Events processed')    


#%% Add info in the classification sheets results 


# Add columns to the all group predictions df
         
classification_results_df = pd.DataFrame(all_group_predictions)


classification_results_df['Prediction'] = classification_results_df['Species'] == classification_results_df['Final_Classification']


classification_results_df.to_csv(os.path.join(results_output_folder, "ALL_GROUP_CLASSIFICATION_SUMMARY.csv"), index=False)         
   
        

#Include coluns with results from logo
 
logo_results_df = pd.DataFrame(logo_probs)

# Step 1: Round columns 3 to 9 (Python is 0-based index, so columns 3:9 = index 2 to 8)
cols_to_process = logo_results_df.columns[3:10]
logo_results_df[cols_to_process] = logo_results_df[cols_to_process].round(3)

# Step 2: Find max value in each row for those columns
logo_results_df['Max_Score'] = logo_results_df[cols_to_process].max(axis=1)

# Step 3: Check for ambiguity (more than one max)
logo_results_df['Ambiguity'] = logo_results_df[cols_to_process].eq(logo_results_df['Max_Score'], axis=0).sum(axis=1).gt(1).map({True: 'yes', False: 'no'})

# Step 4: Find second max value
def second_largest(row):
    unique_vals = sorted(set(row), reverse=True)
    return unique_vals[1] if len(unique_vals) > 1 else unique_vals[0]

logo_results_df['Second_Max_Value'] = logo_results_df[cols_to_process].apply(second_largest, axis=1)

# Step 5: Get column name of max and second max
def get_max_col(row):
    max_val = row.max()
    return row[row == max_val].index.tolist()[0]  # first max if tie

def get_second_max_col(row):
    sorted_vals = row.sort_values(ascending=False)
    if sorted_vals.iloc[0] == sorted_vals.iloc[1]:
        return sorted_vals[sorted_vals == sorted_vals.iloc[0]].index.tolist()[1]  # second of the tied maxes
    else:
        return sorted_vals.index[1]

logo_results_df['Max_Col'] = logo_results_df[cols_to_process].apply(get_max_col, axis=1)
logo_results_df['Second_Max_Col'] = logo_results_df[cols_to_process].apply(get_second_max_col, axis=1)

logo_results_df['Prediction'] = logo_results_df['Species'] == logo_results_df['Max_Col']

logo_results_df.to_csv(os.path.join(results_output_folder, "LOGO_CLASSIFICATION_SUMMARY.csv"), index=False) 

#Include coluns with results from meta
meta_results_df = pd.DataFrame(meta_probs)
 
# Step 1: Round columns 3 to 9 (Python is 0-based index, so columns 3:9 = index 2 to 8)
cols_to_process = meta_results_df.columns[3:10]
meta_results_df[cols_to_process] = meta_results_df[cols_to_process].round(3)

# Step 2: Find max value in each row for those columns
meta_results_df['Max_Score'] = meta_results_df[cols_to_process].max(axis=1)

# Step 3: Check for ambiguity (more than one max)
meta_results_df['Ambiguity'] = meta_results_df[cols_to_process].eq(meta_results_df['Max_Score'], axis=0).sum(axis=1).gt(1).map({True: 'yes', False: 'no'})

# Step 4: Find second max value
def second_largest(row):
    unique_vals = sorted(set(row), reverse=True)
    return unique_vals[1] if len(unique_vals) > 1 else unique_vals[0]

meta_results_df['Second_Max_Value'] = meta_results_df[cols_to_process].apply(second_largest, axis=1)

# Step 5: Get column name of max and second max
def get_max_col(row):
    max_val = row.max()
    return row[row == max_val].index.tolist()[0]  # first max if tie

def get_second_max_col(row):
    sorted_vals = row.sort_values(ascending=False)
    if sorted_vals.iloc[0] == sorted_vals.iloc[1]:
        return sorted_vals[sorted_vals == sorted_vals.iloc[0]].index.tolist()[1]  # second of the tied maxes
    else:
        return sorted_vals.index[1]

meta_results_df['Max_Col'] = meta_results_df[cols_to_process].apply(get_max_col, axis=1)
meta_results_df['Second_Max_Col'] = meta_results_df[cols_to_process].apply(get_second_max_col, axis=1)

meta_results_df['Prediction'] = meta_results_df['Species'] == meta_results_df['Max_Col']


meta_results_df.to_csv(os.path.join(results_output_folder, "META_CLASSIFICATION_SUMMARY.csv"), index=False) 
